# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [14]:
import pandas as pd
import numpy as np
from __future__ import annotations

---
## 2. Load data

In [15]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [16]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [17]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [18]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

### Below are practical features to help churn prediction:

- **Basic cleanup / types**  
  - Copy data, convert date columns to datetime, create `has_gas_flag`.

- **Date-based features**  
  - Create tenure and recency features: `tenure_days`, `days_since_modif`, days-to-renewal features, plus `activ_year`, `activ_month`, `renewal_month`.

- **Consumption features**  
  - Create usage features like `avg_monthly_cons_12m`, last month vs average, total consumption, and `gas_share_12m`.

- **Forecast mismatch features**  
  - Create forecast error features: `forecast_error_12m`, absolute error, percent error, and `forecast_year_minus_12m`.

- **Margin / value features**  
  - Create margin-per-consumption and gap features: `net_margin_per_cons`, `gross_margin_per_cons`, `margin_gap`.

- **Price structure spreads / ratios**  
  - Create peak vs off-peak and mid vs off-peak spreads/ratios (spread, ratio, abs spread).

- **Price change: 6m vs 1y**  
  - Create price change features using diff and percent change for many price columns (`chg_*_year_minus_6m`, `chg_*_pct`).

- **Simple interactions**  
  - Create cross features like `powmax_x_offpeak_price`, `antig_x_products`.

- **Log transforms**  
  - Reduce skew using `log1p_` transforms for consumption and margin related columns.

- **Missing value handling**  
  - Fill numeric NA with median, categorical NA with mode (or “unknown”).

- **One-hot encoding + final cleanup**  
  - One-hot encode categorical columns (`channel_sales`, `origin_up`, `has_gas`), then drop raw date columns for ML readiness.


In [19]:
def feature_engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # -----------------------
    # 1) Basic cleanup / types
    # -----------------------
    date_cols = ["date_activ", "date_end", "date_modif_prod", "date_renewal"]
    for c in date_cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    # Binary flag (keep original too)
    if "has_gas" in df.columns:
        df["has_gas_flag"] = (
            df["has_gas"]
            .map({"t": 1, "f": 0, "T": 1, "F": 0, True: 1, False: 0})
            .astype("float")
        )

    # -----------------------
    # 2) Date-based features
    # -----------------------
    if {"date_activ", "date_end"}.issubset(df.columns):
        df["tenure_days"] = (df["date_end"] - df["date_activ"]).dt.days

    if {"date_end", "date_modif_prod"}.issubset(df.columns):
        df["days_since_modif"] = (df["date_end"] - df["date_modif_prod"]).dt.days

    if {"date_renewal", "date_end"}.issubset(df.columns):
        df["days_to_renewal_from_end"] = (df["date_renewal"] - df["date_end"]).dt.days

    if {"date_renewal", "date_modif_prod"}.issubset(df.columns):
        df["days_to_renewal_from_modif"] = (df["date_renewal"] - df["date_modif_prod"]).dt.days

    # Simple calendar parts (seasonality)
    if "date_activ" in df.columns:
        df["activ_year"] = df["date_activ"].dt.year
        df["activ_month"] = df["date_activ"].dt.month
    if "date_renewal" in df.columns:
        df["renewal_month"] = df["date_renewal"].dt.month

    # -----------------------
    # 3) Consumption features
    # -----------------------
    if "cons_12m" in df.columns:
        df["avg_monthly_cons_12m"] = df["cons_12m"] / 12.0

    if {"cons_last_month", "cons_12m"}.issubset(df.columns):
        df["cons_last_month_vs_avg"] = df["cons_last_month"] / (df["avg_monthly_cons_12m"] + 1e-6)
        df["cons_last_month_minus_avg"] = df["cons_last_month"] - df["avg_monthly_cons_12m"]

    if {"cons_12m", "cons_gas_12m"}.issubset(df.columns):
        total = df["cons_12m"] + df["cons_gas_12m"]
        df["total_cons_12m"] = total
        df["gas_share_12m"] = np.where(total > 0, df["cons_gas_12m"] / total, 0.0)

    # -----------------------
    # 4) Forecast mismatch features
    # -----------------------
    if {"forecast_cons_12m", "cons_12m"}.issubset(df.columns):
        df["forecast_error_12m"] = df["forecast_cons_12m"] - df["cons_12m"]
        df["forecast_abs_error_12m"] = df["forecast_error_12m"].abs()
        df["forecast_abs_error_pct_12m"] = df["forecast_abs_error_12m"] / (df["cons_12m"].abs() + 1.0)

    if {"forecast_cons_year", "cons_12m"}.issubset(df.columns):
        df["forecast_year_minus_12m"] = df["forecast_cons_year"] - df["cons_12m"]

    # -----------------------
    # 5) Margin / value features
    # -----------------------
    if {"net_margin", "total_cons_12m"}.issubset(df.columns):
        df["net_margin_per_cons"] = df["net_margin"] / (df["total_cons_12m"] + 1.0)

    if {"margin_gross_pow_ele", "total_cons_12m"}.issubset(df.columns):
        df["gross_margin_per_cons"] = df["margin_gross_pow_ele"] / (df["total_cons_12m"] + 1.0)

    if {"margin_net_pow_ele", "margin_gross_pow_ele"}.issubset(df.columns):
        df["margin_gap"] = df["margin_net_pow_ele"] - df["margin_gross_pow_ele"]

    # -----------------------
    # 6) Price structure: spreads / ratios
    # -----------------------
    def add_spread_ratio(peak_col: str, off_col: str, out_prefix: str) -> None:
        if {peak_col, off_col}.issubset(df.columns):
            spread = df[peak_col] - df[off_col]
            df[f"{out_prefix}_spread"] = spread
            df[f"{out_prefix}_ratio"] = df[peak_col] / (df[off_col] + 1e-6)
            df[f"{out_prefix}_abs_spread"] = spread.abs()

    add_spread_ratio("forecast_price_energy_peak", "forecast_price_energy_off_peak", "fcast_energy_peak_off")
    add_spread_ratio("var_year_price_peak", "var_year_price_off_peak", "year_price_peak_off")
    add_spread_ratio("var_6m_price_peak", "var_6m_price_off_peak", "m6_price_peak_off")
    add_spread_ratio("var_year_price_mid_peak", "var_year_price_off_peak", "year_price_mid_off")
    add_spread_ratio("var_6m_price_mid_peak", "var_6m_price_off_peak", "m6_price_mid_off")

    # -----------------------
    # 7) Price change: 6m vs 1y (diff and %)
    # -----------------------
    price_pairs = [
        ("var_year_price_off_peak_var", "var_6m_price_off_peak_var", "off_peak_var"),
        ("var_year_price_peak_var", "var_6m_price_peak_var", "peak_var"),
        ("var_year_price_mid_peak_var", "var_6m_price_mid_peak_var", "mid_peak_var"),
        ("var_year_price_off_peak_fix", "var_6m_price_off_peak_fix", "off_peak_fix"),
        ("var_year_price_peak_fix", "var_6m_price_peak_fix", "peak_fix"),
        ("var_year_price_mid_peak_fix", "var_6m_price_mid_peak_fix", "mid_peak_fix"),
        ("var_year_price_off_peak", "var_6m_price_off_peak", "off_peak_total"),
        ("var_year_price_peak", "var_6m_price_peak", "peak_total"),
        ("var_year_price_mid_peak", "var_6m_price_mid_peak", "mid_peak_total"),
    ]

    for year_col, m6_col, name in price_pairs:
        if {year_col, m6_col}.issubset(df.columns):
            df[f"chg_{name}_year_minus_6m"] = df[year_col] - df[m6_col]
            df[f"chg_{name}_pct"] = (df[year_col] - df[m6_col]) / (df[m6_col].abs() + 1e-6)

    # -----------------------
    # 8) Simple interactions
    # -----------------------
    if {"pow_max", "var_year_price_off_peak"}.issubset(df.columns):
        df["powmax_x_offpeak_price"] = df["pow_max"] * df["var_year_price_off_peak"]

    if {"num_years_antig", "nb_prod_act"}.issubset(df.columns):
        df["antig_x_products"] = df["num_years_antig"] * df["nb_prod_act"]

    # -----------------------
    # 9) Log transforms (reduce skew)
    # -----------------------
    for c in ["cons_12m", "cons_gas_12m", "cons_last_month", "net_margin", "imp_cons", "total_cons_12m"]:
        if c in df.columns:
            df[f"log1p_{c}"] = np.log1p(np.maximum(df[c], 0))

    # -----------------------
    # 10) Missing value handling
    # -----------------------
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    cat_cols = df.select_dtypes(include=["object"]).columns
    for c in cat_cols:
        if df[c].isna().any():
            mode = df[c].mode(dropna=True)
            df[c] = df[c].fillna(mode.iloc[0] if not mode.empty else "unknown")

    # -----------------------
    # 11) One-hot encode categoricals for modeling
    # -----------------------
    ohe_cols = [c for c in ["channel_sales", "origin_up", "has_gas"] if c in df.columns]
    df_model = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype=int)

    # Drop raw date columns (avoid datetime issues in ML pipelines)
    for c in date_cols:
        if c in df_model.columns:
            df_model = df_model.drop(columns=[c])

    return df_model

In [20]:
# Save engineered dataset for modeling
OUTPUT_PATH = "engineered_data_for_model.csv"

df_fe = feature_engineer(df)
df_fe.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH} | shape={df_fe.shape}")

Saved: engineered_data_for_model.csv | shape=(14606, 111)
